# Phase 3: Review + Tips Embeddings (Week 9)

Generate Sentence-BERT embeddings for review text + tips and aggregate to user/destination profiles.

Using both reviews and tips provides richer semantic information about what users and destinations are like.

In [ ]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
from config import *
from src.data.cache import DataCache
from src.data.loader import load_reviews, load_tips
from src.models.embedding_utils import ReviewTipsEmbedder

cache = DataCache(str(CACHE_DIR))
businesses = cache.load_dataframe('businesses_major_cities')
reviews = cache.load_dataframe('reviews_cross_city')

## Load Tips Data

In [ ]:
# Load tips (sample 10% same as reviews)
from config import RAW_DATA_DIR, PHASE1_SAMPLE_FRACTION
tips_file = RAW_DATA_DIR / 'yelp_academic_dataset_tip.json'

tips = load_tips(str(tips_file), sample=True, sample_fraction=PHASE1_SAMPLE_FRACTION)
print(f"Loaded {len(tips):,} tips")
print(f"Tips sample:")
print(tips.head(2))

## Initialize Embedder

In [ ]:
# Initialize embedder (downloads model on first run)
print("Loading Sentence-BERT model (first time: ~500MB download)...")
embedder = ReviewTipsEmbedder(model_name='all-MiniLM-L6-v2')
print(f"✓ Model loaded. Embedding dimension: 384")

## Generate User Embeddings

In [ ]:
# Generate embeddings for sample users
user_embeddings = {}
sample_users = reviews['user_id'].unique()[:100]  # Sample first 100 users

for i, user_id in enumerate(sample_users):
    if i % 20 == 0:
        print(f"  Processing user {i+1}/100...")

    embedding = embedder.aggregate_user_embeddings(user_id, reviews, tips, businesses)
    if embedding is not None:
        user_embeddings[user_id] = embedding

print(f"✓ Generated embeddings for {len(user_embeddings)} users")

## Generate Destination Embeddings

In [ ]:
# Generate embeddings for each city
dest_embeddings = {}
cities = businesses['city'].unique()

for city in cities:
    embedding = embedder.aggregate_destination_embeddings(city, reviews, tips, businesses)
    if embedding is not None:
        dest_embeddings[city] = embedding

print(f"✓ Generated embeddings for {len(dest_embeddings)} cities")
print(f"Cities: {list(dest_embeddings.keys())[:5]}...")

## Cache Embeddings

In [ ]:
# Cache embeddings
cache.save_embeddings(user_embeddings, 'user_embeddings')
cache.save_embeddings(dest_embeddings, 'destination_embeddings')

print("✓ Embeddings cached")
print("Ready for Phase 4: Machine Learning Models")